# Optimal policy estimation with MonteCarlo

Consider the MDP carachterized by $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$.

We consider the common case where the probabilities of transition are unknown, i.e. $\mathcal{P}:=\{P^{a}:\, a\in\mathcal{A}\}$ is unknown.

>A common approach to approximate the optimal policy consists in alternating the following two steps:
>1) estimating the value function by MonteCarlo;
>2) updating the policy by chosing the action that maximizes the value function.

**The MonteCarlo estimation of the value function**:

Under the assumptions of a stationary policy, consider the value function

$$
v_{\pi}(s):=\mathbb{E}_{\pi} \left[ G_{0} \mid S_{0}=s \right],
$$

where $G_{0}:=\sum\limits_{k=0}^{T \wedge \tau} \gamma^{k} R_{1+k} $ is the *objective function* at time $0$, with $R_{1+k}$ the reward at time $1+k$, $T$ a finite horizon and $\tau$ the minimum time to reach an absorbing state. The objective function selects a random trajectory starting from state $s$ under the policy $\pi$.

Let $G_{1},\dots,G_{n}$ be a sample of i.i.d. $n$ objective functions distributed as $G_{0}$, then the MonteCarlo estimator of the value function $v_{\pi}$ at time $n\in\mathbb{N}^{+}$ is the empirical average:

$$
\hat{v}_{\pi,n}(s) := \frac{1}{n} \sum\limits_{i=1}^{n} G_{i}(s).
$$

In [5]:
import numpy as np

In [6]:
def objective_function(R_all, pi, P_trans_allow, s0, gamma, T, absorbing_state=None):
    '''
    Computes the objective function G(s)
    by selecting a random trajectory,
    until T steps or an absorbing state is reached.

    R_all[a, s, s'] : matrix of rewards obtained if action a is applied to state s and the next state is s'
    pi[s, a] : policy, i.e. probability of choosing action a being in state s
    P_trans_allow[a, s, s'] : matrices P_{s,s'}^{a} with 1 where the transition is eventually allowed
                          : transition rates are not known!!
    s0: initial state to compute G(s0)
    gamma: discount factor
    T : horizon / length of a trajectory
    absorbing_state: list of absorbing states known by the expert

    output : objective function G(s0) for a random trajectory starting from s0
    '''

    if absorbing_state is None:    # if no absorbing states are provided, set empty list
        absorbing_state = []

    G = 0                        # objective function counter
    t = 0                        # time step counter
    s = s0                       # initial state

    n_states = R_all.shape[1]                            # number of states
    n_actions = R_all.shape[0]                           # number of actions

    while s not in absorbing_state and t < T:

        # Randomly choose new action to apply to s, according to policy
        new_action = np.random.choice(n_actions, p=pi[s])

        # Save possible next states s'
        possible_next_states = np.flatnonzero(P_trans_allow[new_action, s] > 0) 

        # check if there is at least one possible next state
        if possible_next_states.size == 0:         
            raise ValueError(
                f"No allowed next state for state {s}, action {new_action}. "
                "Each row of P_trans_allow[a, s, :] must contain at least one non-zero entry."
            )

        # Randomly choose next state s' among the available ones 
        s_new = np.random.choice(possible_next_states)

        # Update objective function
        G += (gamma ** t) * R_all[new_action, s, s_new] 

        # Update steps
        t += 1
        # Update state
        s = s_new

    return G


In [7]:
def value_function_MC(R_all, pi, P_trans_allow, gamma, T, N, absorbing_state=None):
    '''
    Computes the value function v(s) for a given policy pi
    by the Monte Carlo method

    R_all[a, s, s'] : matrix of rewards obtained if action a is applied to state s and the next state is s'
    pi[s, a] : policy
    P_trans_allow[a,s,s'] : matrices P_{s,s'}^{a} with 1 where the transition is eventually allowed
                          : transition rates are not known!!
    gamma: discount factor
    T : horizon / length of a trajectory
    N : number of trajectories
    absorbing_states: list of absorbing states known by the expert

    output: value function v_pi(s) estimated by Monte Carlo method
    '''

    # Save N rewards from N possible realisations, i.e. G_1, G_2, ..., G_N
    n_states = R_all.shape[1]             # number of states
    rewards = np.zeros(n_states)          # store rewards for each state

    for s in range(n_states):             # generate N trajectories from each state s
        for _ in range(N):
            
            rewards[s] += objective_function(R_all, pi, P_trans_allow, s, gamma, T, absorbing_state)
  
    # Monte Carlo estimation
    return rewards / N


**A remark on the code:**

- For a state $s$, the action to apply $a$, is chosen under the policy $\pi(a\mid s)$.

- For a state $s$ and action $a$, the new state $s'$ is randomly unformly chosen from the matrix of allowed transitions, under the condition *P_trans_allowed $[a,s,s'] > 0 $.*

- For a state $s$, action $a$, new state $s'$, the reward is given by matrix of rewards R_all[a,s,s'].

- This model assumes not known the transition probabilities to choose a state $s'$ starting from a state $s$ and an action $a$. Anyways, it takes into account some information about the environment: the way states are possibly connected under the choice of all actions.

### Example

Toy example from the course.

In [8]:
N = 1000      # number of trajectories
T = 100         # horizon / length of a trajectory

# states = [high, low]
# actions = [s, w, c]

R_s = np.array([[5, 5], [-10, 5]])
R_w = np.array([[1, 0], [0, 1]])
R_c = np.array([[0, 0], [0, 0]])
R_complet = np.stack([R_s, R_w, R_c], axis=0) 
# matrix of rewards

P_s_allowed = np.array([[1, 1], [1, 1]])          
P_w_allowed = np.array([[1, 0], [0, 1]])
P_c_allowed = np.array([[1 ,0], [1 ,0]])
P_act_allowed = np.stack([P_s_allowed, P_w_allowed, P_c_allowed], axis=0)
# environmental matrix P[a,s,s'] with 1 where the transition is eventually allowed

pi_policy_s_w = np.array([[1, 0, 0], [0, 1, 0]])
pi_policy_s_s = np.array([[1, 0, 0], [1, 0, 0]])
pi_policy_s_c = np.array([[1, 0, 0], [0, 0, 1]])
# 3 cases of policy

gamma_ex = 0.8

print("politique s/w : ", value_function_MC(R_complet, pi_policy_s_w, P_act, gamma=gamma_ex, T=T, N=N))
print("politique s/s : ", value_function_MC(R_complet, pi_policy_s_s, P_act, gamma=gamma_ex, T=T, N=N))
print("politique s/c : ", value_function_MC(R_complet, pi_policy_s_c, P_act, gamma=gamma_ex, T=T, N=N))

NameError: name 'P_act' is not defined

## Algorithme MC online $\epsilon$-greedy

In [ ]:
def trajectory(pi, P_trans_allow, s0, T, absorbing_state=None):
    '''
    Simulates a trajectory of T steps on states and actions.

    pi[s, a] : policy, i.e. probability of choosing action a being in state s
    P_trans_allow[a,s,s_prime] : 1 where the transition s -> s_prime is allowed under action a
                                : transition probabilities are not known (uniform draw among allowed states)
    s0 : initial state
    T : number of steps
    absorbing_state : list of absorbing states known by the expert (trajectory stops there)

    output : states_trajectory, actions_trajectory
        states_trajectory : list of visited states [s_0, s_1, ..., s_T]  (length T+1)
        actions_trajectory : list of actions taken  [a_0, a_1, ..., a_{T-1}] (length T)
    '''

    if absorbing_state is None:
        absorbing_state = []

    n_states, n_actions = pi.shape

    states_trajectory = []         # store states trajectory
    actions_trajectory = []        # store actions trajectory

    s = s0
    states_trajectory.append(s)

    for _ in range(T):

        # Stop if the current state is absorbing
        if s in absorbing_state:
            break

        # Choose an action according to the policy
        new_a = np.random.choice(n_actions, p=pi[s])
        actions_trajectory.append(new_a)

        # Draw the next state uniformly among the allowed ones
        possible_next_states = np.flatnonzero(P_trans_allow[new_a, s, :] > 0)
        if len(possible_next_states) == 0:
            raise ValueError(
                f"No allowed next state for state {s}, action {new_a}. "
                "Each row of P_trans_allow[a, s, :] must contain at least one non-zero entry."
            )

        s_new = np.random.choice(possible_next_states)
        states_trajectory.append(s_new)

        # restart from new state
        s = s_new

    return states_trajectory, actions_trajectory

In [ ]:
def online_MC_greedy(R_1, pi_0, P_trans_allow, gamma, T, N, epsilon, q_0, absorbing_state=None):
  '''
    On-line Monte Carlo control with an epsilon-greedy policy (every-visit).
    The initial state of each episode is drawn uniformly at random.

    R_1[a,s] : matrix of rewards weighted by probabilities
    pi_0[s, a] : initial policy (probability of choosing a in s)
    P_trans_allow[a, s, s_prime] : 1 where the transition is allowed
    gamma : discount factor
    T : length of an episode
    N : number of episodes
    epsilon : exploration rate in [0,1]
    q_0 : initial values of states/actions value function q_value
    absorbing_state : list of absorbing states known by the expert

    output : q_value : states/actions value function
           : pi      : optimal policy
    '''

  if absorbing_state is None:
    absorbing_state = []

  n_states, n_actions = pi_0.shape

  # non absorbing states
  non_absorbing = [ s for s in range(n_states) if s not in absorbing_state]  
  if len(non_absorbing) == 0:
    raise ValueError("No non-absorbing states.")

  # Check dimensions
  assert R_1.shape == (n_states, n_actions)
  assert len(q_0) == n_states

  # initial policy
  pi = pi_0.astype(float).copy()                                     

  # Initialize Q(s,a) = q_0(s)
  q_value = np.tile(np.asarray(q_0, dtype=float)[:, None], (1, n_actions))

  # Number of returns observed for each (s,a)
  counts = np.zeros((n_states, n_actions))    

  for _ in range(N):

    # 1. Draw a random initial state non absorbing and generate a trajectory until tau ^ T
    s0 = np.random.choice(non_absorbing)
    traj_s, traj_a = trajectory(pi, P_trans_allow, s0, T, absorbing_state)

    # 2. Compute rewards backwards: G_t = r_t + gamma * G_{t+1}
    objective_G = 0

    for t in reversed(range(len(traj_a))):   

      s, a = traj_s[t], traj_a[t]
      objective_G = R_1[s, a] + gamma * objective_G

      # Every-visit incremental mean
      counts[s, a] += 1

      # q_value computation
      q_value[s, a] += (objective_G - q_value[s, a]) / counts[s, a]


    # 3. Policy improvement: epsilon-greedy with respect to Q
    visited_states = set(traj_s[:-1])

    for s in visited_states:
      
      # Identify all maximizing actions
      best_actions = np.flatnonzero(q_value[s] == np.max(q_value[s]))

      # Distribute exploration probability
      pi[s, :] = epsilon / n_actions

      # Distribute exploitation probability among ties
      pi[s, best_actions] += ( (1 - epsilon) / len(best_actions))


    return q_value, pi


### Example:

In [ ]:
N = 1000     # number of trajectories
T = 100      # horizon
eps = 0.1
gamma = 0.8

q_0 = [0, 0]

# states = [high, low]
# actions = [s, w, c]

R_1 = np.array([[5 ,1 ,0], [ -1 ,1 ,0]])
# matrix of rewards weighted by probabilities

P_s_allowed = np.array([[1, 1], [1, 1]])          
P_w_allowed = np.array([[1, 0], [0, 1]])
P_c_allowed = np.array([[1 ,0], [1 ,0]])
P_act_allowed = np.stack([P_s_allowed, P_w_allowed, P_c_allowed], axis=0)
# environmental matrix P[a,s,s'] with 1 where the transition is eventually allowed

pi_policy_0 = np.array([[0.5, 0.3, 0.2], [0.1, 0, 0.9]])
# initial policy


In [ ]:
q1, pi1 = online_MC_greedy(R_1, pi_policy_0, P_act_allowed, gamma=gamma, T=T, N=N, epsilon=eps, q_0=q_0)
print("states/actions optimal function:\n", q1, "\n optimal policy:\n", pi1)


states/actions optimal function:
 [[13.169006    9.89122983  8.34526285]
 [ 4.13959716  0.          9.52228761]] 
 optimal policy:
 [[0.93333333 0.03333333 0.03333333]
 [0.03333333 0.03333333 0.93333333]]


## Assumptions: $P_{s,s'}^{a}$ are known

To compare the results of the example, here we assume that the probabilities of transition are known. The new state s' is chosen under the matrix of transition probabilities $P_{s,s'}^{a}$, instaed than randomly uniformly.

In [ ]:
import numpy as np

In [ ]:
def objective_function(R_all, pi, P_act, s0, gamma, T, absorbing_state=None):
    '''
    Computes the objective function G(s)
    by selecting a random trajectory,
    until T steps or an absorbing state is reached.

    R_all[a, s, s'] : matrix of rewards obtained if action a is applied to state s and the next state is s'
    pi[s, a] : policy, i.e. probability of choosing action a being in state s
    P_act[a, s, s'] : matrices P_{s,s'}^{a} with probabilities of transition
    s0: initial state to compute G(s0)
    gamma: discount factor
    T : horizon / length of a trajectory
    absorbing_state: list of absorbing states known by the expert

    output : objective function G(s0) for a random trajectory starting from s0
    '''

    if absorbing_state is None:    # if no absorbing states are provided, set empty list
        absorbing_state = []

    G = 0                        # objective function counter
    t = 0                        # time step counter
    s = s0                       # initial state

    n_states = R_all.shape[1]                            # number of states
    n_actions = R_all.shape[0]                           # number of actions

    while s not in absorbing_state and t < T:

        # Randomly choose new action to apply to s, according to policy
        new_action = np.random.choice(n_actions, p=pi[s])

        # Randomly choose next state under the probabilities of transition
        s_new = np.random.choice(n_states, p=P_act[new_action, s])

        # Update objective function
        G += (gamma ** t) * R_all[new_action, s, s_new] 

        # Update steps
        t += 1
        # Update state
        s = s_new

    return G


In [ ]:
def value_function_MC(R_all, pi, P_act, gamma, T, N, absorbing_state=None):
    '''
    Computes the value function v(s) for a given policy pi
    by the Monte Carlo method

    R_all[a, s, s'] : matrix of rewards obtained if action a is applied to state s and the next state is s'
    pi[s, a] : policy
    P_act[a,s,s'] : matrices P_{s,s'}^{a} of transition probabilities
    gamma: discount factor
    T : horizon / length of a trajectory
    N : number of trajectories
    absorbing_states: list of absorbing states known by the expert

    output: value function v_pi(s) estimated by Monte Carlo method
    '''

    # Save N rewards from N possible realisations, i.e. G_1, G_2, ..., G_N
    n_states = R_all.shape[1]             # number of states
    rewards = np.zeros(n_states)          # store rewards for each state

    for s in range(n_states):             # generate N trajectories from each state s
        for _ in range(N):
            
            rewards[s] += objective_function(R_all, pi, P_act, s, gamma, T, absorbing_state)
  
    # Monte Carlo estimation
    return rewards / N


In [ ]:
N = 10000       # number of trajectories
T = 100         # horizon / length of a trajectory

# states = [high, low]
# actions = [s, w, c]

R_s = np.array([[5, 5], [-10, 5]])
R_w = np.array([[1, 0], [0, 1]])
R_c = np.array([[0, 0], [0, 0]])
R_complet = np.stack([R_s, R_w, R_c], axis=0) 
# matrix of rewards

P_s = np.array([[0.3,0.7],[0.4,0.6]])
P_w = np.eye(2)
P_c = np.array([[1,0],[1,0]])
P_act = np.stack([P_s,P_w,P_c], axis=0)
# P[a][s][s'] matrix of transition probabilites

pi_policy_s_w = np.array([[1, 0, 0], [0, 1, 0]])
pi_policy_s_s = np.array([[1, 0, 0], [1, 0, 0]])
pi_policy_s_c = np.array([[1, 0, 0], [0, 0, 1]])
# 3 cases of policy

gamma_ex = 0.8

print("politique s/w : ", value_function_MC(R_complet, pi_policy_s_w, P_act, gamma=gamma_ex, T=T, N=N))
print("politique s/s : ", value_function_MC(R_complet, pi_policy_s_s, P_act, gamma=gamma_ex, T=T, N=N))
print("politique s/c : ", value_function_MC(R_complet, pi_policy_s_c, P_act, gamma=gamma_ex, T=T, N=N))

politique s/w :  [10.26147541  5.        ]
politique s/s :  [9.4466745  3.83880587]
politique s/c :  [16.03046269 12.82105339]


### Algorithme MC online $\epsilon$-greedy

In [11]:
def trajectory(pi, P_act, s0, T, absorbing_state=None):
    '''
    Simulates a trajectory of T steps on states and actions.

    pi[s, a] : policy, i.e. probability of choosing action a being in state s
    P_act[a,s,s'] : matrix of transition probabilities from s to s' with a
    s0 : initial state
    T : number of steps
    absorbing_state : list of absorbing states known by the expert (trajectory stops there)

    output : states_trajectory, actions_trajectory
        states_trajectory : list of visited states [s_0, s_1, ..., s_T]  (length T+1)
        actions_trajectory : list of actions taken  [a_0, a_1, ..., a_{T-1}] (length T)
    '''

    if absorbing_state is None:
        absorbing_state = []

    n_states, n_actions = pi.shape

    states_trajectory = []         # store states trajectory
    actions_trajectory = []        # store actions trajectory

    s = s0
    states_trajectory.append(s)

    for _ in range(T):

        # Stop if the current state is absorbing
        if s in absorbing_state:
            break

        # Choose an action according to the policy
        new_a = np.random.choice(n_actions, p=pi[s])
        actions_trajectory.append(new_a)

        # Choose an action according to the policy
        s_new = np.random.choice(n_states, p=P_act[new_a, s])
        states_trajectory.append(s_new)

        # restart from new state
        s = s_new

    return states_trajectory, actions_trajectory

In [ ]:
def online_MC_greedy(R_1, pi_0, P_act, gamma, T, N, epsilon, q_0, absorbing_state=None):
  '''
    On-line Monte Carlo control with an epsilon-greedy policy (every-visit).
    The initial state of each episode is drawn uniformly at random.

    R_1[a,s] : matrix of rewards weighted by probabilities
    pi_0[s, a] : initial policy (probability of choosing a in s)
    P_act[a,s,s'] : matrix of transition probabilities from s to s' with a
    gamma : discount factor
    T : length of an episode
    N : number of episodes
    epsilon : exploration rate in [0,1]
    q_0 : initial values of states/actions value function q_value
    absorbing_state : list of absorbing states known by the expert

    output : q_value : states/actions value function
           : pi      : optimal policy
    '''

  if absorbing_state is None:
    absorbing_state = []

  n_states, n_actions = pi_0.shape

  # non absorbing states
  non_absorbing = [ s for s in range(n_states) if s not in absorbing_state]  
  if len(non_absorbing) == 0:
    raise ValueError("No non-absorbing states.")

  # Check dimensions
  assert R_1.shape == (n_states, n_actions)
  assert len(q_0) == n_states

  # initial policy
  pi = pi_0.astype(float).copy()                                     

  # Initialize Q(s,a) = q_0(s)
  q_value = np.tile(np.asarray(q_0, dtype=float)[:, None], (1, n_actions))

  # Number of returns observed for each (s,a)
  n_visits = np.zeros((n_states, n_actions))    

  for _ in range(N):

    # generate a trajectory until tau ^ T, from random s0
    s0 = np.random.choice(non_absorbing)
    traj_s, traj_a = trajectory(pi, P_act, s0, T, absorbing_state)

    # Monte Carlo estimation of Q, sequentially
    objective_G = 0
    for t in reversed(range(len(traj_a))):   

      s, a = traj_s[t], traj_a[t]

      objective_G = R_1[s, a] + gamma * objective_G      # current G

      n_visits[s, a] += 1                                # number of visits

      q_value[s, a] += (objective_G - q_value[s, a]) / n_visits[s, a]  # update q value function


    # Update the policy
    visited_states = set(traj_s[:-1])
    for s in visited_states:    

      best_actions = np.flatnonzero(q_value[s] == np.max(q_value[s]))  # find best actions

      # Distribute exploration probability
      pi[s, :] = epsilon / n_actions

      # Distribute exploitation probability
      pi[s, best_actions] += ( (1 - epsilon) / len(best_actions))


    return q_value, pi


In [12]:
N = 1000     # number of trajectories
T = 100      # horizon
eps = 0.1
gamma = 0.8

q_0 = [0, 0]

# states = [high, low]
# actions = [s, w, c]

R_1 = np.array([[5 ,1 ,0] ,[ -1 ,1 ,0]])
# matrix of rewards weighted by probabilities

P_s = np.array([[0.3,0.7],[0.4,0.6]])
P_w = np.eye(2)
P_c = np.array([[1,0],[1,0]])
P_act = np.stack([P_s,P_w,P_c], axis=0)
# P[a][s][s'] matrix of transition probabilites

pi_policy_0 = np.array([[0.5, 0.3, 0.2], [0.1, 0.5, 0.4]])
# initial policy

In [13]:
q1, pi1 = online_MC_greedy(R_1, pi_policy_0, P_act, gamma=gamma, T=T, N=N, epsilon=eps, q_0=q_0)
print("states/actions optimal function:\n", q1, "\n optimal policy:\n", pi1)


states/actions optimal function:
 [[10.68836142  8.37418524  8.42611388]
 [ 7.52964779  6.44920714  6.41973418]] 
 optimal policy:
 [[0.93333333 0.03333333 0.03333333]
 [0.93333333 0.03333333 0.03333333]]
